In [1]:
# Import libraries
import pycaret
from pycaret.classification import *
from pycaret.utils import enable_colab
enable_colab()
import matplotlib.pyplot as plt
%matplotlib inline
import os
import pandas

Colab mode enabled.


In [2]:
## Download data
if not os.path.isfile('diabetic_data.csv'):
    #http://archive.ics.uci.edu/ml/datasets/Diabetes+130-US+hospitals+for+years+1999-2008#
    url = 'https://docs.google.com/uc?export=download&id=1k5-1caezQ3zWJbKaiMULTGq-3sz6uThC'
    r = requests.get(url, allow_redirects=True)
    open('diabetic_data.csv', 'wb').write(r.content)

In [3]:
# Check some data
df = pd.read_csv('diabetic_data.csv', index_col=False)
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
# Check all columns names
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')

In [5]:
drop_columns = []#['diag_1', 'diag_2', 'diag_3','encounter_id', 'acetohexamide', 'tolbutamide', 'examide', 'citoglipton', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
for col in df.columns:
    if '_id' in col:
        drop_columns.append(col)
drop_columns = [*set(drop_columns)]

In [6]:
df = df.drop(columns=drop_columns)
df.columns

Index(['patient_nbr', 'race', 'gender', 'age', 'weight', 'time_in_hospital',
       'payer_code', 'medical_specialty', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin',
       'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
       'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')

In [7]:
exp_name = setup(data = df,  target = 'diabetesMed', normalize=True, session_id = 42, silent = True, profile=True)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
eda(display_format = "bokeh")#"bokeh" for interactive plots or "svg"

Imported v0.1.58. After importing, execute '%matplotlib inline' to display charts in Jupyter.
    AV = AutoViz_Class()
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)
Update: verbose=0 displays charts in your local Jupyter notebook.
        verbose=1 additionally provides EDA data cleaning suggestions. It also displays charts.
        verbose=2 does not display charts but saves them in AutoViz_Plots folder in local machine.
        chart_format='bokeh' displays charts in your local Jupyter notebook.
        chart_format='server' displays charts in your browser: one tab for each chart type
        chart_format='html' silently saves interactive HTML files in your local machine
Shape of your Data Set loaded: (101766, 46)
#######################################################################################
######################## C L A S 

,Nuniques,dtype,Nulls,Nullpercent,NuniquePercent,Value counts Min,Data cleaning improvement suggestions
patient_nbr,9486,int64,0,0.000000,94.860000,0,
diag_1,469,object,0,0.000000,4.690000,1,combine rare categories
diag_3,464,object,0,0.000000,4.640000,1,combine rare categories
diag_2,460,object,0,0.000000,4.600000,1,combine rare categories
num_lab_procedures,103,int64,0,0.000000,1.030000,0,
num_medications,64,int64,0,0.000000,0.640000,0,
medical_specialty,56,object,0,0.000000,0.560000,1,combine rare categories
number_outpatient,23,int64,0,0.000000,0.230000,0,
payer_code,17,object,0,0.000000,0.170000,4,
number_emergency,16,int64,0,0.000000,0.160000,0,


    45 Predictors classified...
        7 variables removed since they were ID or low-information variables
        List of variables removed: ['acetohexamide', 'tolbutamide', 'examide', 'citoglipton', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

################ Binary_Classification problem #####################
Number of variables = 38 exceeds limit, finding top 30 variables through XGBoost
    No categorical feature reduction done. All 29 Categorical vars selected 
    Removing correlated variables from 9 numerics using SULO method
Selecting all (9) variables since none of them are highly correlated...
    Adding 29 categorical variables to reduced numeric variables  of 9
############## F E A T U R E   S E L E C T I O N  ####################
Current number of predictors = 38 
    Finding Important Features using Boosted Trees algorithm...
        using 38 variables...
Finding top features using XGB is crashing. Continuing with all predictors...


,Nuniques,dtype,Nulls,Nullpercent,NuniquePercent,Value counts Min,Data cleaning improvement suggestions
patient_nbr,71518,int64,0,0.000000,70.276910,0,
diag_3,790,object,0,0.000000,0.776291,1,combine rare categories
diag_2,749,object,0,0.000000,0.736002,1,combine rare categories
diag_1,717,object,0,0.000000,0.704558,1,combine rare categories
num_lab_procedures,118,int64,0,0.000000,0.115952,0,
num_medications,75,int64,0,0.000000,0.073698,0,
medical_specialty,73,object,0,0.000000,0.071733,1,combine rare categories
number_outpatient,39,int64,0,0.000000,0.038323,0,
number_emergency,33,int64,0,0.000000,0.032427,0,
number_inpatient,21,int64,0,0.000000,0.020636,0,


    30 Predictors classified...
    3 variables removed since they were ID or low-information variables
    List of variables removed: ['diag_1', 'diag_2', 'diag_3']
   Total columns > 30, too numerous to print.


Row
    [0] WidgetBox
        [0] Select(name='y', options=['patient_nbr', ...], value='patient_nbr')
    [1] ParamFunction(function)

... (5 outputs left out)

No date vars could be found in data set


HoloViews(Overlay)

Row
    [0] WidgetBox
        [0] Select(name='X-Axis', options=['race', 'gender', ...], value='race')
        [1] Select(name='Y-Axis', options=['patient_nbr', ...], value='patient_nbr')
    [1] ParamFunction(function)

Time to run AutoViz (in seconds) = 18


In [9]:
columns=['examide','citoglipton', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'insulin','change','number_emergency','acetohexamide',
                     'tolbutamide', 'glimepiride-pioglitazone', 'weight','number_emergency','number_outpatient', 'diag_1','diag_2','diag_3','medical_specialty']
df = df.drop(columns=columns)

df.columns

Index(['patient_nbr', 'race', 'gender', 'age', 'time_in_hospital',
       'payer_code', 'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_inpatient', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
       'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
       'glyburide-metformin', 'glipizide-metformin', 'diabetesMed',
       'readmitted'],
      dtype='object')

In [10]:
exp_name = setup(data = df,  target = 'diabetesMed', normalize=True, session_id = 42, silent = True, profile=True)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
eda(display_format = "bokeh")

Shape of your Data Set loaded: (101766, 30)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
Data cleaning improvement suggestions. Complete them before proceeding to ML modeling.


,Nuniques,dtype,Nulls,Nullpercent,NuniquePercent,Value counts Min,Data cleaning improvement suggestions
patient_nbr,9486,int64,0,0.000000,94.860000,0,
num_lab_procedures,103,int64,0,0.000000,1.030000,0,
num_medications,64,int64,0,0.000000,0.640000,0,
payer_code,17,object,0,0.000000,0.170000,4,
number_inpatient,15,int64,0,0.000000,0.150000,0,
time_in_hospital,14,int64,0,0.000000,0.140000,0,
number_diagnoses,12,int64,0,0.000000,0.120000,0,
age,10,object,0,0.000000,0.100000,12,
num_procedures,7,int64,0,0.000000,0.070000,0,
race,6,object,0,0.000000,0.060000,68,


    29 Predictors classified...
        No variables removed since no ID or low-information variables found in data set

################ Binary_Classification problem #####################


Row
    [0] WidgetBox
        [0] Select(name='y', options=['patient_nbr', ...], value='patient_nbr')
    [1] ParamFunction(function)

Row
    [0] WidgetBox
        [0] Select(name='X-Axis', options=['patient_nbr', ...], value='patient_nbr')
        [1] Select(name='Y-Axis', options=['patient_nbr', ...], value='num_lab_procedures')
        [2] Select(name='Color', options=['None', 'diabetesMed'], value='None')
    [1] ParamFunction(function)

Column
    [0] Row
        [0] HoloViews(DynamicMap)
        [1] Column
            [0] WidgetBox
                [0] Select(margin=(20, 20, 20, 20), options=['race', 'age', ...], value='race', width=250)
            [1] VSpacer()

... (3 outputs left out)

No date vars could be found in data set


HoloViews(Overlay)

Row
    [0] WidgetBox
        [0] Select(name='X-Axis', options=['race', 'age', ...], value='race')
        [1] Select(name='Y-Axis', options=['patient_nbr', ...], value='patient_nbr')
    [1] ParamFunction(function)

Time to run AutoViz (in seconds) = 10
